# Image FetchPush conedir — OFFLINE ORIGINAL-STYLE qualification (fixed alpha=0 + BC 0.05 + twin critics)

Strictly-offline (seed 0) qualification of the *original-style* actor on pixels:

```
actor_loss = 0.05 * BC_NLL(data_action | state, future_goal)
           + 0.95 * (-min_Q(state, policy_action, future_goal))
```

`entropy_coefficient=0.0` (fixed alpha=0, no adaptive optimizer) · `bc_coef=0.05` ·
`twin_q=True` (actor uses min(Q1,Q2)) · `random_goals=0.0` (same-trajectory positives) ·
frozen `.npz` IS the buffer (zero env interaction). All eval distances are **physical
object-goal coordinates** (sim), never flattened image-L2.

Pipeline: pinned install → clone+checkout → mount Drive → env/render audit →
**collect-or-reuse the frozen dataset** → `scripts/qualify_image_offline_original_style`
(preflight → 100k offline → physical fixed-eval → checkpoint diagnostics → GIFs → verdict).
Writes to a NEW run dir; never overwrites prior runs.

> ⚠️ Requires the instrumentation + driver from commit **c6c5e3f** (or later). Set `COMMIT` in
> cell 2 accordingly.

In [ ]:
# 1. Dependencies WITHOUT disturbing Colab's preinstalled GPU JAX (proven-safe recipe).
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"                         # headless render backend on Colab
import jax, jaxlib, numpy, subprocess, sys
hold = [f"jax=={jax.__version__}", f"jaxlib=={jaxlib.__version__}", f"numpy=={numpy.__version__}"]
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
print("Colab JAX", jax.__version__, "| devices:", jax.devices())
pip("--no-deps","dm-haiku","optax","chex")
pip("jmp","tabulate","toolz","etils","tensorboardX","gymnasium-robotics","mujoco","imageio-ffmpeg",*hold)
print("post-install JAX", jax.__version__, "| devices:", jax.devices())

In [ ]:
# 2. Clone the fork and checkout the commit that CONTAINS the instrumentation + driver.
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
COMMIT = "c6c5e3f"   # instrumentation + qualification driver
!git fetch -q origin && git checkout -q $COMMIT
!git log -1 --oneline
assert os.path.exists("scripts/qualify_image_offline_original_style.py"),     "checkout is MISSING the qualification driver -- set COMMIT to c6c5e3f or later."
assert os.path.exists("scripts/collect_push_dataset.py"), "missing dataset collector."

In [ ]:
# 3. Mount Google Drive (checkpoints + artifacts saved here).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. Paths. Frozen dataset is COLLECTED ONCE (cell 6), then reused. New run dir.
import os, json, numpy as np
ENV  = "fetch_push_image_conedir"
SEED = 0
SMOKE = True     # True = tiny integration smoke; set False for the real 100k qualification

BASE = "/content/drive/MyDrive/easypush_image_qual"
DATA = f"{BASE}/data/push_image_conedir_noisy_oracle_s{SEED}" + ("_smoke" if SMOKE else "") + ".npz"
RUN_DIR = f"{BASE}/fetch_push_image_conedir_off_bc005_alpha0_twin_s{SEED}" + ("_smoke" if SMOKE else "")
OUT = "artifacts/image_conedir_offline_original_style"
STEPS = 2000 if SMOKE else 100000
N_EPISODES = 50 if SMOKE else 1000
assert not os.path.exists(os.path.join(RUN_DIR, "final.pkl")),     f"{RUN_DIR} already has a finished run; choose a fresh dir rather than overwrite."
print("DATA   :", DATA, "(exists)" if os.path.exists(DATA) else "(will collect)")
print("RUN_DIR:", RUN_DIR)

In [ ]:
# 5. Environment/render audit -- STOP before anything if it fails.
!python -m scripts.smoke_image_conedir
audit = json.load(open("artifacts/push_image_probe/smoke_image_conedir.json"))
assert audit["verdict"] == "PASS", "image-conedir env audit FAILED -- stopping."
print("env audit PASS (oracle=%.2f random=%.2f)" % (
    audit["gate4_oracle"]["oracle_success"], audit["gate4_oracle"]["random_success"]))

In [ ]:
# 6. Collect the FROZEN dataset (ONCE; reused if it already exists). NOT a download --
#    it renders the FetchPush image env with a noisy scripted oracle + a random-episode
#    fraction. ~15s for 50 smoke episodes / ~5 min for 1000 on a T4.
os.makedirs(os.path.dirname(DATA), exist_ok=True)
if os.path.exists(DATA):
    meta = json.loads(str(np.load(DATA, allow_pickle=True)["meta"]))
    print("reusing existing dataset:", DATA)
else:
    from scripts.collect_push_dataset import collect
    print(f"collecting {N_EPISODES} episodes -> {DATA}")
    meta = collect(ENV, episodes=N_EPISODES, noise=0.3, random_frac=0.2, seed=SEED, out=DATA)
print(json.dumps(meta, indent=2))
assert meta["behavior_success_oracle"] and meta["behavior_success_oracle"] >= 0.9,     "demonstrator broken -- oracle arm should succeed >=0.9"
assert os.path.exists(DATA), "dataset collection failed"
print("dataset ready, behavior_success =", meta["behavior_success"])

In [ ]:
# 7. Run the qualification driver: preflight -> 100k offline -> physical fixed-eval ->
#    checkpoint diagnostics (20k/50k/100k) -> rollout GIFs -> verdict. Physical metrics only.
#    Resume-safe: re-run with the same RUN_DIR + --resume to continue from latest.pkl.
cmd = ["python","-m","scripts.qualify_image_offline_original_style",
       "--dataset", DATA, "--run_dir", RUN_DIR, "--out", OUT,
       "--seed", str(SEED), "--steps", str(STEPS)]
if SMOKE: cmd.append("--smoke")
print(" ".join(cmd))
import subprocess
subprocess.run(cmd, check=True)

In [ ]:
# 8. Show results: preflight, verdict, physical fixed-eval, checkpoint diagnostics, GIFs.
import json, os
from IPython.display import Image, display
print("=== PREFLIGHT ==="); print(json.dumps(json.load(open(f"{OUT}/preflight_audit.json"))["checks"], indent=2))
summ = json.load(open(f"{OUT}/summary.json"))
print("
=== VERDICT:", summ["qualification_verdict"], "===")
print(json.dumps(summ["final_checkpoint_physical"], indent=2))
print("
=== fixed_eval.csv ==="); print(open(f"{OUT}/fixed_eval.csv").read())
print("=== checkpoint_diagnostics.csv ==="); print(open(f"{OUT}/checkpoint_diagnostics.csv").read())
for s in ("20000","50000","100000"):
    g = f"{RUN_DIR}/rollout_{s}.gif"
    if os.path.exists(g): display(Image(g, width=256))